# Robustness and supplementary analyses

This notebook documents the sensitivity checks described in the manuscript. It contains code only: the restricted analytical records, saved outputs, and result tables are not included.


## Re-estimate models without 2022 publications

Repeat the main impact models after removing publications from 2022 to assess possible effects of incomplete coverage and shorter citation windows.


In [ ]:
df_2 = df_1[df_1['year']<2022]

In [ ]:
df_jl = pd.read_csv('sciscinet2\gu_data_c5_normalized_20251122.csv')
df_ac1 = pd.read_parquet("matched_rows.parquet")
df_ac1 = df_ac1.drop_duplicates(subset=['paperid','doi'])[['PMID','paperid']]

In [ ]:
df_merge_0 = df_2.merge(df_ac1, left_on='PMID', right_on='PMID', how='inner')
df_merged = df_merge_0.merge(df_jl, left_on='paperid', right_on='paper_id', how='inner')

In [ ]:
df_citation = df_merged[['PMID', 'year', 'matched_IDs','matched_count','human','animal','inst_type', 'funding_subtype', 'country_type', 'country_type_v2','mesh_lvl1','team_size_group','document_type_group', 'mesh_lvl2','c5_norm']]
#df_citation

In [ ]:
top1_cutoff = df_citation['c5_norm'].quantile(0.99)
top5_cutoff = df_citation['c5_norm'].quantile(0.95)
top10_cutoff = df_citation['c5_norm'].quantile(0.90)

mean_c5 = df_citation['c5_norm'].mean()


df_citation['top1_pct'] = (
    df_citation['c5_norm'] >= top1_cutoff
).astype(int)

df_citation['top5_pct'] = (
    df_citation['c5_norm'] >= top5_cutoff
).astype(int)

df_citation['top10_pct'] = (
    df_citation['c5_norm'] >= top10_cutoff
).astype(int)

df_citation['above_avg'] = (
    df_citation['c5_norm'] > mean_c5
).astype(int)

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_citation_j = pd.merge(df_citation, d_journal_2, on='PMID', how='inner')

In [ ]:
d_journali = pd.read_csv('a3_JIF.csv',encoding='utf-8').drop_duplicates()
d_journal_2 = d_journali[['id', 'journalName', 'jcrYear', 'abbrJournal',
       'category','totalArticles', 'jif2019', 'jif5Years',
       'averageJifPercantage', 'jci', 'quartile',
       'immediacyIndex','jifPercentile','title', 'yearPublished', 'jciPercentile','aisQuartile']]

In [ ]:
d_journal_3 = d_journali[['id', 'journalName', 'jcrYear','jifPercentile','quartile']].dropna(subset=['jifPercentile']).copy()
#d_journal_3['q_num'] = d_journal_3['quartile'].str.replace('Q', '').astype(int)

conditions = [
    d_journal_3['jifPercentile'] > 75,
    d_journal_3['jifPercentile'] > 50,
    d_journal_3['jifPercentile'] > 25,
    d_journal_3['jifPercentile'] >= 0
]

choices = [1, 2, 3, 4]

d_journal_3['q_num'] = np.select(
    conditions,
    choices,
    default=np.nan
).astype(int)

bins = [0,10,20,30,40,50,60,70,80,90,100]
labels = [10,9,8,7,6,5,4,3,2,1]

d_journal_3['decile'] = pd.cut(
    d_journal_3['jifPercentile'],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

In [ ]:
d_journal_3['journal_match'] = (
    d_journal_3['journalName']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df_citation_j['journal_match'] = (
    df_citation_j['source_title']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# merge
df_citation_j2 = df_citation_j.merge(
    d_journal_3[['journal_match', 'jcrYear', 'jifPercentile', 'q_num', 'decile']],
    left_on=['journal_match', 'year'],
    right_on=['journal_match', 'jcrYear'],
    how='left'
)


df_citation_j2.drop(columns=['journal_match', 'jcrYear'], inplace=True)

In [ ]:
df_citation_j3 = df_citation_j2[
    df_citation_j2['jifPercentile'].notna()
].copy()

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'c5_norm ~ C(inst_type, Treatment(reference="public")) + C(year)+C(q_num)+C(team_size_group)+C(document_type_group)',
    data=df_citation_j3
).fit()

print(model.summary())

In [ ]:
import statsmodels.api as sm
import patsy
formula = "top10_pct ~ C(inst_type, Treatment(reference='public'))+ C(year)+C(q_num)+C(team_size_group)+C(document_type_group)"  

y, X = patsy.dmatrices(formula, df_citation_j3, return_type="dataframe")

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
)

res = model.fit()
print(res.summary())

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_2, d_journal_2, on='PMID', how='inner')
df_cem_p = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype', '2017_sum', '2018_sum', '2019_sum', '2020_sum', '2021_sum', '2022_sum','total_sum',"team_size_group","document_type_group"]]

In [ ]:
df_cem_p['inst_type'] = df_cem_p['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
df_cem_p["year_group"] = pd.cut(
    df_cem_p["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
df = df_cem_p.copy()
df["treat"] = (df["total_sum"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_p = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_p = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_p,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_p.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = [
    "PMID", "treat", "weight", "stratum_id",
    'matched_count','human','animal',
    "year", "year_group", "source_title",
    "team_size_group", "document_type_group",
    "inst_type",'funding_subtype', "mesh_lvl1", "total_sum"
]

keep_cols = [
    c for c in keep_cols
    if c in df_treat.columns and c in df_control_long.columns
]

df_matched_long_bal_p = pd.concat(
    [
        df_treat[keep_cols],
        df_control_long[keep_cols]
    ],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_pp = pd.read_csv("PKG\C16_Link_Patents_Papers.tsv", encoding="utf-8", sep="\t")
df_ppc = df_pp.drop_duplicates(['PMID','PatentId'])[['PMID','PatentId']].groupby('PMID').count().reset_index()
df_final_pattern = df_2.merge(df_ppc, on="PMID", how="left")
df_final_pattern["PatentId"] = df_final_pattern["PatentId"].fillna(0).astype(int)
df_final_pattern["has_patent"] = (df_final_pattern["PatentId"] > 0).astype(int)

In [ ]:
df_final_pattern['inst_type'] = df_final_pattern['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_pattern, d_journal_2, on='PMID', how='inner')
df_cem_pp = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_patent","team_size_group","document_type_group"]]

In [ ]:
df_cem_pp["year_group"] = pd.cut(
    df_cem_pp["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
df_cem_pp['has_patent'].value_counts()

In [ ]:
df = df_cem_pp.copy()
df["treat"] = (df["has_patent"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_pp = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_pp = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_pp,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_pp.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_patent","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_pp = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_pc = pd.read_csv("PKG\C12_Link_Papers_Clinicaltrials.tsv", encoding="utf-8", sep="\t")
df_pcc = df_pc.drop_duplicates(['PMID','nct_id'])[['PMID','nct_id']].groupby('PMID').count().reset_index()
df_final_clinical = df_2.merge(df_pcc, on="PMID", how="left")
df_final_clinical["nct_id"] = df_final_clinical["nct_id"].fillna(0).astype(int)
df_final_clinical["has_CT"] = (df_final_clinical["nct_id"] > 0).astype(int)

In [ ]:
df_final_clinical['inst_type'] = df_final_clinical['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_clinical, d_journal_2, on='PMID', how='inner')
df_cem_c = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_CT","team_size_group","document_type_group"]]

In [ ]:
df_cem_c["year_group"] = pd.cut(
    df_cem_c["year"],
    bins=[2004, 2007, 2010, 2013, 2016, 2019, 2022],
    labels=[1, 2, 3, 4, 5, 6]
).astype(int)

In [ ]:
df = df_cem_c.copy()
df["treat"] = (df["has_CT"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_c = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_c = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_c,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_c.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_CT","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_c = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference='public'))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())

## Check small disease areas

Identify papers assigned exclusively to disease areas with fewer than 10,000 publications and assess how many records would be removed by this restriction.


In [ ]:
small_diseases = {"C21", "C24"}

def keep_after_disease_filter(mesh_list):
    return any(code not in small_diseases for code in mesh_list)

df_3 = df_1[
    df_1["mesh_lvl1"].apply(keep_after_disease_filter)
].copy()

print("Original N:", len(df_1))
print("Robustness N:", len(df_3))
print("Dropped N:", len(df_1) - len(df_3))

## Compare papers with exactly two funders

Restrict the cohort to papers with exactly two identified funders, then repeat the impact models to compare funding structures at the same funder count.


In [ ]:
df_4 = df_1[df_1["matched_count"] == 2].copy()

print("Original N:", len(df_1))
print("Restricted N:", len(df_4))
print("Share:", len(df_4) / len(df_1))

print("\nFunding structure:")
print(df_4["inst_type"].value_counts())
print("\nPercentage:")
print(df_4["inst_type"].value_counts(normalize=True) * 100)

In [ ]:
df_jl = pd.read_csv('sciscinet2\gu_data_c5_normalized_20251122.csv')
df_ac1 = pd.read_parquet("matched_rows.parquet")
df_ac1 = df_ac1.drop_duplicates(subset=['paperid','doi'])[['PMID','paperid']]

In [ ]:
df_merge_0 = df_4.merge(df_ac1, left_on='PMID', right_on='PMID', how='inner')
df_merged = df_merge_0.merge(df_jl, left_on='paperid', right_on='paper_id', how='inner')

In [ ]:
df_citation = df_merged[['PMID', 'year', 'matched_IDs','matched_count','human','animal','inst_type', 'funding_subtype', 'country_type', 'country_type_v2','mesh_lvl1','team_size_group','document_type_group', 'mesh_lvl2','c5_norm']]
#df_citation

In [ ]:
top1_cutoff = df_citation['c5_norm'].quantile(0.99)
top5_cutoff = df_citation['c5_norm'].quantile(0.95)
top10_cutoff = df_citation['c5_norm'].quantile(0.90)

mean_c5 = df_citation['c5_norm'].mean()


df_citation['top1_pct'] = (
    df_citation['c5_norm'] >= top1_cutoff
).astype(int)

df_citation['top5_pct'] = (
    df_citation['c5_norm'] >= top5_cutoff
).astype(int)

df_citation['top10_pct'] = (
    df_citation['c5_norm'] >= top10_cutoff
).astype(int)

df_citation['above_avg'] = (
    df_citation['c5_norm'] > mean_c5
).astype(int)

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_citation_j = pd.merge(df_citation, d_journal_2, on='PMID', how='inner')

In [ ]:
d_journali = pd.read_csv('a3_JIF.csv',encoding='utf-8').drop_duplicates()
d_journal_2 = d_journali[['id', 'journalName', 'jcrYear', 'abbrJournal',
       'category','totalArticles', 'jif2019', 'jif5Years',
       'averageJifPercantage', 'jci', 'quartile',
       'immediacyIndex','jifPercentile','title', 'yearPublished', 'jciPercentile','aisQuartile']]

In [ ]:
d_journal_3 = d_journali[['id', 'journalName', 'jcrYear','jifPercentile','quartile']].dropna(subset=['jifPercentile']).copy()
#d_journal_3['q_num'] = d_journal_3['quartile'].str.replace('Q', '').astype(int)

conditions = [
    d_journal_3['jifPercentile'] > 75,
    d_journal_3['jifPercentile'] > 50,
    d_journal_3['jifPercentile'] > 25,
    d_journal_3['jifPercentile'] >= 0
]

choices = [1, 2, 3, 4]

d_journal_3['q_num'] = np.select(
    conditions,
    choices,
    default=np.nan
).astype(int)

bins = [0,10,20,30,40,50,60,70,80,90,100]
labels = [10,9,8,7,6,5,4,3,2,1]

d_journal_3['decile'] = pd.cut(
    d_journal_3['jifPercentile'],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

In [ ]:
d_journal_3['journal_match'] = (
    d_journal_3['journalName']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df_citation_j['journal_match'] = (
    df_citation_j['source_title']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# merge
df_citation_j2 = df_citation_j.merge(
    d_journal_3[['journal_match', 'jcrYear', 'jifPercentile', 'q_num', 'decile']],
    left_on=['journal_match', 'year'],
    right_on=['journal_match', 'jcrYear'],
    how='left'
)


df_citation_j2.drop(columns=['journal_match', 'jcrYear'], inplace=True)

In [ ]:
df_citation_j3 = df_citation_j2[
    df_citation_j2['jifPercentile'].notna()
].copy()

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'c5_norm ~ C(inst_type, Treatment(reference="public")) + C(year)+C(q_num)+C(team_size_group)+C(document_type_group)',
    data=df_citation_j3
).fit()

print(model.summary())

In [ ]:
import statsmodels.api as sm
import patsy
formula = "top10_pct ~ C(inst_type, Treatment(reference='public'))+ C(year)+C(q_num)+C(team_size_group)+C(document_type_group)"  

y, X = patsy.dmatrices(formula, df_citation_j3, return_type="dataframe")

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
)

res = model.fit()
print(res.summary())

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_4, d_journal_2, on='PMID', how='inner')
df_cem_p = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype', '2017_sum', '2018_sum', '2019_sum', '2020_sum', '2021_sum', '2022_sum','total_sum',"team_size_group","document_type_group"]]

In [ ]:
df_cem_p['inst_type'] = df_cem_p['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
df_cem_p["year_group"] = pd.cut(
    df_cem_p["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
df = df_cem_p.copy()
df["treat"] = (df["total_sum"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_p = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_p = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_p,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_p.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = [
    "PMID", "treat", "weight", "stratum_id",
    'matched_count','human','animal',
    "year", "year_group", "source_title",
    "team_size_group", "document_type_group",
    "inst_type",'funding_subtype', "mesh_lvl1", "total_sum"
]

keep_cols = [
    c for c in keep_cols
    if c in df_treat.columns and c in df_control_long.columns
]

df_matched_long_bal_p = pd.concat(
    [
        df_treat[keep_cols],
        df_control_long[keep_cols]
    ],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_pp = pd.read_csv("PKG\C16_Link_Patents_Papers.tsv", encoding="utf-8", sep="\t")
df_ppc = df_pp.drop_duplicates(['PMID','PatentId'])[['PMID','PatentId']].groupby('PMID').count().reset_index()
df_final_pattern = df_4.merge(df_ppc, on="PMID", how="left")
df_final_pattern["PatentId"] = df_final_pattern["PatentId"].fillna(0).astype(int)
df_final_pattern["has_patent"] = (df_final_pattern["PatentId"] > 0).astype(int)

In [ ]:
df_final_pattern['inst_type'] = df_final_pattern['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_pattern, d_journal_2, on='PMID', how='inner')
df_cem_pp = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_patent","team_size_group","document_type_group"]]

In [ ]:
df_cem_pp["year_group"] = pd.cut(
    df_cem_pp["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
df_cem_pp['has_patent'].value_counts()

In [ ]:
df = df_cem_pp.copy()
df["treat"] = (df["has_patent"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_pp = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_pp = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_pp,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_pp.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_patent","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_pp = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_pc = pd.read_csv("PKG\C12_Link_Papers_Clinicaltrials.tsv", encoding="utf-8", sep="\t")
df_pcc = df_pc.drop_duplicates(['PMID','nct_id'])[['PMID','nct_id']].groupby('PMID').count().reset_index()
df_final_clinical = df_4.merge(df_pcc, on="PMID", how="left")
df_final_clinical["nct_id"] = df_final_clinical["nct_id"].fillna(0).astype(int)
df_final_clinical["has_CT"] = (df_final_clinical["nct_id"] > 0).astype(int)

In [ ]:
df_final_clinical['inst_type'] = df_final_clinical['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_clinical, d_journal_2, on='PMID', how='inner')
df_cem_c = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_CT","team_size_group","document_type_group"]]

In [ ]:
df_cem_c["year_group"] = pd.cut(
    df_cem_c["year"],
    bins=[2004, 2007, 2010, 2013, 2016, 2019, 2022],
    labels=[1, 2, 3, 4, 5, 6]
).astype(int)

In [ ]:
df = df_cem_c.copy()
df["treat"] = (df["has_CT"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_c = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_c = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_c,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_c.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_CT","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_c = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference='public'))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())

## Validate funding-agency classification

Compare automatically assigned agency categories with manually coded organizations and calculate classification accuracy. The organization-level source files are not distributed.


In [ ]:
data1 = pd.read_json("chatgpt_agency/agency_classification_eq5_v2.json", lines=True)
data2 = pd.read_excel('chatgpt_agency/2_merged_agency_classification_eq5_1000.xlsx')

In [ ]:
data1.merge(data2[['original_agency', 'preferred_name', 'catogroy_label']], on=['original_agency', 'preferred_name'], how='inner')

In [ ]:
df = data1.merge(data2[['original_agency', 'preferred_name', 'catogroy_label']], on=['original_agency', 'preferred_name'], how='inner')

cm = pd.crosstab(df['catogroy_label'], df['category'], rownames=['Manual classification'], colnames=['LLM classification'])
display(cm)

accuracy = (df['category'] == df['catogroy_label']).mean()
print(f'Total: {len(df):,}')
print(f'Accuracy: {accuracy:.2%}')
print(f'Correct: {(df["category"] == df["catogroy_label"]).sum():,}')